<a href="https://colab.research.google.com/github/outsider8946/maxidom-hack/blob/dolor/maxidom_hack_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Init

In [ ]:
!pip install "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0" -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json

raw = json.load(open('/content/products_final.json'))
data = raw['products']

In [ ]:
import numpy as np
cats = [list(item['categories'].values())[0] for item in data]
un_cats = np.unique(cats)

len(un_cats), len(cats)

(2375, 88790)

In [ ]:
from frida_decisions import Judge

judge = Judge.from_pretrained("ai-forever/FRIDA-Decisions", device="cuda")

request = {
    "state": "Здравствуйте, у меня не приходит код подтверждения уже час.",
    "questions": {
        "topic": {
            "type": "choice",
            "instructions": "К какой теме относится обращение?",
            "criteria": {
                "login": "вход в аккаунт, коды подтверждения, пароли",
                "payment": "оплата, списания, возвраты",
                "delivery": "доставка заказа",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "Оцени срочность обращения.",
            "criteria": ["низкая", "средняя", "высокая"],
        },
        "spam": {
            "type": "noul",
            "instructions": "Является ли сообщение спамом?",
            "criteria": {"true": "реклама, чужие ссылки, просьба перевести деньги",
                         "false": "вопрос или жалоба по нашему сервису"},
        },
    },
}
print(judge.judge(request)["answers"])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

{'topic': {'type': 'choice', 'probabilities': {'login': 0.9988761817604521, 'payment': 0.0005957733860450987, 'delivery': 0.0005280448535027795}, 'confidence': 0.9913236073016475, 'choice': 'login'}, 'urgency': {'type': 'score', 'probabilities': {'0': 0.5741066157175307, '1': 0.08303048096154746, '2': 0.34286290332092173}, 'confidence': 0.18785816363148822, 'score': 0.7687562876033909, 'legend': {'0': 'низкая', '1': 'средняя', '2': 'высокая'}}, 'spam': {'type': 'noul', 'noul': 0.0030366369886944116}}


# DB annotate (unrelevant filter)

In [ ]:
def get_noul_request(cat):
  return {
      "state": f"Категория каталога: {cat}",
      "questions": {
          "constructions": {
              "type": "noul",
              "instructions": "Относится ли эта категория товаров к строительству, ремонту или отделке помещений?",
              "criteria": {
                  "true": (
                      "стройматериалы и отделка: плитка, керамогранит, ламинат, линолеум, "
                      "краска, эмаль, грунтовка, обои, ГКЛ, штукатурка, шпаклёвка, "
                      "сухие смеси, цемент, кирпич, утеплитель, гидроизоляция, "
                      "электрика (кабель, розетки, автоматы), сантехника (трубы, смесители), "
                      "плинтусы, наличники, двери, окна, крепёж, строительный инструмент"
                  ),
                  "false": (
                      "не относится к стройке: растения и кашпо, косметика и гигиена, "
                      "одежда и обувь, зоотовары, продукты питания, мебель, "
                      "декор и текстиль, игрушки, канцтовары, автотовары, сад и дача, "
                      "бытовая техника, электроника"
                  ),
              },
          }
      }
  }


def get_choice_request(cat, samples):
  samples_descr = []
  for sample in samples:
    descr = sample.get('marketing_description') or sample.get('name')
    samples_descr.append(descr)

  state = (
        f"Категория каталога: {cat}\n"
        f"Примеры товаров из каталога:\n{'\n'.join(samples_descr)}"
    )
  return {
      "state": state,
      "questions": {
          "construction": {
              "type": "choice",
              "instructions": "Как эта категория относится к ремонту помещений?",
              "criteria": {
                  "core": "материалы и работы: плитка, краска, обои, ГКЛ, кабель, розетки, автоматы, трубы, смесители, сухие смеси, плинтусы, светильники для монтажа",
                  "adjacent": "сопутствующее ремонту: инструмент, крепёж, расходники, встраиваемая техника, кондиционеры, бойлеры, вентиляция, спецодежда, защита",
                  "excluded": "не относится к ремонту: отдельно стоящая бытовая техника, электроника, растения, косметика, одежда, зоо, продукты, мебель, декор, игрушки",
              },
          }
      }
  }


In [ ]:
target_cat = 'Трафареты на окна'
products = []
for item in data:
  catalog_cat = list(item['categories'].values())[0]
  if catalog_cat == target_cat:
    products.append(item)

len(products)

2

In [ ]:
products

[{'id': 107733,
  'cod_good': '1000739842',
  'name': 'трафарет Новогодний 44,5х59,5см белый в асс-те',
  'marketing_description': None,
  'url_key': '/catalog/trafarety-na-okna/1000739842/',
  'price': 0,
  'currency': 'RUB',
  'brand': {'id': 37, 'name': 'KAEMINGK'},
  'collection': None,
  'country': {'id': 1881, 'name': 'Китай'},
  'categories': {'225': 'Трафареты на окна'},
  'unit': None,
  'description': 'Название: Новогодний<br>Размер трафарета: 44,5х59,5<br>Материал: бумага<br>Количество в наборе: 6<br>Цвет: белый',
  'weight': None,
  'characteristics': []},
 {'id': 107736,
  'cod_good': '1000794132',
  'name': 'трафарет Снежинки 21х30см белый',
  'marketing_description': None,
  'url_key': '/catalog/trafarety-na-okna/1000794132/',
  'price': 0,
  'currency': 'RUB',
  'brand': {'id': 37, 'name': 'KAEMINGK'},
  'collection': None,
  'country': {'id': 1881, 'name': 'Китай'},
  'categories': {'225': 'Трафареты на окна'},
  'unit': None,
  'description': 'Название: Снежинки<br>Ра

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def medoids(products, k=3):
    texts = [f"{p['name']} {(p.get('description') or '')[:100]}" for p in products]
    X = TfidfVectorizer().fit_transform(texts)
    S = cosine_similarity(X)
    # medoid = строка с максимальной суммой сходств
    scores = S.sum(axis=1)
    idx = np.argsort(scores)[::-1][:k]
    return [products[i] for i in idx]

In [ ]:
def get_medoids_from_cat(target_cat):
  products = []
  for item in data:
    catalog_cat = list(item['categories'].values())[0]
    if catalog_cat == target_cat:
      products.append(item)
  return medoids(products)

In [ ]:
from tqdm import tqdm

def get_requests_batch(cats):
  medoids_batch = []

  for cat in tqdm(cats, desc='Computing medioids...'):
    medoids_batch.append(get_medoids_from_cat(cat))

  requests_batch = []
  for cat, medoids in tqdm(zip(cats, medoids_batch), desc='Formatting requests...'):
    requests_batch.append(get_choice_request(cat, medoids) )

  return requests_batch

In [ ]:
sample_cats = np.random.choice(un_cats, size=10)
sample_cats

array(['Наборы ключей', 'Замки накладные', 'Кресла плетеные',
       'Обогреватели масляные', 'Файлы', 'Модули памяти', 'Батарейки',
       'Шкафы для посуды', 'Средства для умывания лица',
       'Полотна к ножовкам'], dtype='<U60')

In [ ]:
requests_batch = get_requests_batch(sample_cats)

Computing medioids...: 100%|██████████| 10/10 [00:00<00:00, 14.70it/s]
Formatting requests...: 10it [00:00, 41817.59it/s]


In [ ]:
out_batch = judge.judge_batch(requests_batch)

In [ ]:
requests_batch[-1]

{'state': 'Категория каталога: Детекторы\nПримеры товаров из каталога:\nПодходит для поиска по металлу и пластику. Максимальная глубина поиска: 3 см.\n3 режима: обнаружение металла, проводов под напряжением, деревянного каркаса. Автоматическое отключение для экономии заряда батареи. Удобный паз для маркировки места обнаружения объекта. Подсветка для работы в слабоосвещенных местах.\nДетекторы Navigator используются для поиска металла и проводки внутри стен и конструкций. Приборы позволяют обнаружить находящиеся под напряжением и без напряжения электрические провода, металл, а также деревянный каркас. Звуковая и световая индикации. Небольшой вес. Эргономичная компактная конструкция из АБС-пластика. 2 режима: обнаружение металла и обнаружение проводки. Регулировка уровня чувствительности с помощью колесика.',
 'questions': {'construction': {'type': 'choice',
   'instructions': 'Как эта категория относится к ремонту помещений?',
   'criteria': {'core': 'материалы и работы: плитка, краска,

In [ ]:
out_batch[-1]

{'model': 'FRIDA-Decisions',
 'answers': {'construction': {'type': 'choice',
   'probabilities': {'core': 0.22711833990110344,
    'adjacent': 0.6172621544775828,
    'excluded': 0.15561950562131374},
   'confidence': 0.15897127282554235,
   'choice': 'adjacent'}},
 'margins': {'construction': {'core': 2.7536349296569824,
   'adjacent': 3.753457546234131,
   'excluded': 2.375577688217163}},
 'usage': {'state_tokens': 165,
  'state_truncated': False,
  'questions': 1,
  'candidates': 3,
  'rows': 10,
  'encoder_tokens': 4142,
  'state_cache': 'off',
  'backend': 'torch',
  'milliseconds': 1158.57}}

In [ ]:
requests_batch = get_requests_batch(un_cats)

Computing medioids...: 100%|██████████| 2375/2375 [02:28<00:00, 15.95it/s]
Formatting requests...: 2375it [00:00, 105879.62it/s]


In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
import torch
from tqdm import tqdm

def postprocessing_batch(out_batch):
  outs = []
  for out in out_batch:
    probs = out['answers']['construction']['probabilities']
    label = max(probs, key=probs.get)
    outs.append(label)

  return outs

def run_batch(requests_batch, batch_size = 32):
  outs = []
  n = len(requests_batch)
  total_batches = (n + batch_size - 1) // batch_size
  pbar = tqdm(range(0, n, batch_size), total=total_batches, desc="Judging", unit="batch")
  for i in pbar:
     batch = requests_batch[i:i + batch_size]
     outs_judge_batch = judge.judge_batch(batch)
     outs_batch = postprocessing_batch(outs_judge_batch)
     outs.extend(outs_batch)
     torch.cuda.empty_cache()
  return outs

In [ ]:
out_batch = run_batch(requests_batch)

Judging: 100%|██████████| 75/75 [04:06<00:00,  3.29s/batch]


In [ ]:
out_batch[0]

'excluded'

In [ ]:
catalog_mapping = {
    str(cat): label for cat, label in zip(un_cats, out_batch)
}

In [ ]:
import numpy as np

ul, uc = np.unique(out_batch, return_counts=True)

for l, c in zip(ul, uc):
  print(f'{l}: {c}')

adjacent: 752
core: 617
excluded: 1006


In [ ]:
import json

with open('catalog_mapping_filtering.json', 'w', encoding='utf-8') as f:
  json.dump(catalog_mapping, f, indent=4, ensure_ascii=False)